In [ ]:
#!pip install crewai

  Using cached lark-1.3.1-py3-none-any.whl.metadata (1.8 kB)
  Using cached overrides-7.7.0-py3-none-any.whl.metadata (5.8 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 26.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 28.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.0/42.0 MB 46.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 48.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 40.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.3/18.3 MB 50.0 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 47.7 MB/s  0:00:00
Using cached lark-1.3.1-py3-none-any.whl (113 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 48.3 MB/s  0:00:00 eta 0:00:01
Using cached overrides-7.7.0-py3-none-any.whl (17 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5

In [ ]:
#!pip install 'crewai[tools]'

  Using cached defusedxml-0.7.1-py2.py3-none-any.whl.metadata (32 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 841.5/841.5 kB 22.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.5/22.5 MB 41.6 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 995.3/995.3 kB 21.2 MB/s  0:00:00
Using cached defusedxml-0.7.1-py2.py3-none-any.whl (25 kB)
  Attempting uninstall: tiktokenm━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/7 [pymupdf]
    Found existing installation: tiktoken 0.14.0━━━━━━━━━━━━━━ 2/7 [pymupdf]
    Uninstalling tiktoken-0.14.0:m━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/7 [pymupdf]
      Successfully uninstalled tiktoken-0.14.0━━━━━━━━━━━━━━━━ 2/7 [pymupdf]
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [crewai-tools] [crewai-tools]


In [7]:
import os
from crewai import Agent,Task, Crew, Process

from crewai_tools import SerperDevTool
from crewai import LLM

from dotenv import load_dotenv
load_dotenv()

search_tool = SerperDevTool()

research_agent = Agent(
    role="Research Agent",
    goal="Perform research tasks using the SerperDev tool.",
    backstory="You are a research agent with extensive experience in performing research tasks using the SerperDev tool.",
    verbose=True,
    allow_delegation=False,
    llm=LLM(model="gpt-6.1-sol", temperature=0),
    tools=[search_tool]
)   
write_agent = Agent(
    role="Write Agent",
    goal="Write a comprehensive report based on the research findings.",
    backstory="You are a skilled writer with expertise in creating comprehensive reports based on research findings.",
    verbose=True,
    allow_delegation=False,
    llm=LLM(model="gpt-6.1-sol", temperature=0),
    tools=[]
)


In [ ]:
research_task = Task(
    name="Research",
    description="Perform research on {topic} using the SerperDev tool.",
    expected_output="A bullet-point summary of the key findings, with source URLs.",
    agent=research_agent
)

write_task = Task(
    name="Write Report",
    description="Write a comprehensive report on {topic} based on the research findings.",
    expected_output="A well-structured report in markdown with an introduction, key sections, and a conclusion.",
    agent=write_agent,
    context=[research_task]
)

In [ ]:
crew = Crew(
    agents=[research_agent, write_agent],
    tasks=[research_task, write_task],
    process=Process.sequential,
    verbose=True
)

result = crew.kickoff(inputs={"topic": "Agentic AI trends in 2026"})
print(result)